# Attention as a Weighted Estimator

This notebook shows how transformer attention parallels familiar econometric estimators. We:

- derive attention as an adaptive kernel smoother,
- compare softmax weights to weighted least squares (WLS) weights,
- visualise the alignment between both weighting schemes on a real sentence.

> Tip: Run the notebook after installing the packages in `requirements.txt`.


## Learning goals

1. Re-express scaled dot-product attention as a kernel smoother.
2. Compare attention weights with WLS weights on a toy heteroskedastic dataset.
3. Interpret the resulting outputs as conditional expectations.


In [ ]:
import math
from dataclasses import dataclass
from typing import Dict, Tuple

import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 1. A heteroskedastic toy dataset

We simulate a regression problem where later observations are noisier. WLS is well-motivated: we downweight the high-variance points. Attention will treat tokens with high similarity as low-variance contributors, echoing the same instinct.


In [ ]:
rng = np.random.default_rng(123)
n_obs = 12
x = np.linspace(-2, 2, n_obs)
noise_scale = np.linspace(0.3, 1.2, n_obs)
y = 1.2 * x + 0.4 + rng.normal(scale=noise_scale)
weights = 1 / (noise_scale ** 2)

wls_df = pd.DataFrame({"x": x, "y": y, "weight": weights})
wls_df

## 2. Fit WLS and inspect weights

We solve the closed-form WLS estimator and plot both the regression fit and the weights to remind ourselves that economics already trusts weighted estimators.


In [ ]:
X = np.column_stack([np.ones_like(x), x])
W = np.diag(weights)
beta_hat = np.linalg.inv(X.T @ W @ X) @ X.T @ W @ y
beta_hat

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(x, y, s=weights * 50, c=weights, cmap="viridis")
ax[0].plot(x, X @ beta_hat, color="black", lw=2)
ax[0].set_title("Weighted least squares fit")
ax[0].set_xlabel("x")
ax[0].set_ylabel("y")

ax[1].stem(range(n_obs), weights, use_line_collection=True)
ax[1].set_title("Observation weights")
ax[1].set_xlabel("Observation")
ax[1].set_ylabel("Weight")

plt.tight_layout()
plt.show()


## 3. Compute attention weights on the same sentence

We prompt a small GPT-2 variant with the sentence below and extract attention weights from the final layer. The query token plays the role of the evaluation point `x`, and dot products deliver similarity-based weights—just like kernels.


In [ ]:
MODEL_NAME = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True).to(DEVICE)

sentence = "Inflation targeting requires credible forward guidance from the central bank."
inputs = tokenizer(sentence, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    outputs = model(**inputs)
    attn = outputs.attentions[-1][0]  # last layer, first head stack

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
attn_matrix = attn.mean(dim=0).cpu().numpy()  # mean across heads
attn_matrix.shape


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
heat = sns.heatmap(attn_matrix, xticklabels=tokens, yticklabels=tokens, cmap="magma", ax=ax)
ax.set_title("Attention weights (mean over heads)")
plt.xticks(rotation=70)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
target_token_index = tokens.index("forward")
weights_attention = attn_matrix[target_token_index]
comparison_df = pd.DataFrame({
    "token": tokens,
    "attention_weight": weights_attention,
    "normalized_wls": weights / weights.sum()
})
comparison_df.head()


In [ ]:
comparison_df.set_index("token").plot.bar(figsize=(10, 4))
plt.title("Attention vs. WLS-inspired weights")
plt.ylabel("Weight")
plt.tight_layout()
plt.show()


## Econometric takeaways

- Softmax temperature behaves like a bandwidth parameter: lowering it sharpens the kernel, focusing on closer tokens.
- Attention weights can be interpreted as `E[V | Q = q]`—a conditional expectation computed in embedding space.
- Comparing attention weights to WLS weights helps audiences see that transformers are sophisticated weighting machines, not inscrutable black boxes.
